# Clasificador de imágenes propio

**Proyecto IPCC – ML.** Mismo recorrido de la clase (datos → modelo → evaluación), pero con un
conjunto de fotos construido por el grupo: **tres objetos** fotografiados con el celular.

Objetos: **OBJETO 1 · OBJETO 2 · OBJETO 3** _(reemplazar por los nombres reales)_

### Cómo se usa
- **Shift + Enter** ejecuta una celda. Ejecutar **en orden** y esperar a que cada una termine.
- Las celdas marcadas **⚙ PARÁMETROS** son las únicas que se tocan para cambiar de carpeta o para
  hacer los experimentos. **Un experimento = cambiar una sola cosa y volver a entrenar.**
- Las fotos van en dos carpetas, una subcarpeta por objeto (el nombre de la subcarpeta es la
  etiqueta):

```
fotos/
   entrenamiento/   (60 por objeto)
      objeto1/  objeto2/  objeto3/
   prueba/          (10 por objeto, tomadas en otras condiciones)
      objeto1/  objeto2/  objeto3/
```

## 0. Preparación del entorno

### Activar la GPU antes de empezar

Una GPU es un procesador diseñado para hacer muchas operaciones simultáneas, que es
exactamente lo que requiere entrenar una red neuronal. Con GPU esta práctica toma unos 3 minutos;
sin ella, más de media hora.

En Colab: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.

La celda siguiente verifica que quedó activa.

In [ ]:
import torch

# cuda es la plataforma de computo en GPU de NVIDIA.
if torch.cuda.is_available():
    print("GPU activa:", torch.cuda.get_device_name(0))
else:
    print("ATENCION: no hay GPU asignada.")
    print("Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> T4 GPU")

Ahora instalamos **fastai**, una biblioteca que permite entrenar modelos de visión por computador
con relativamente poco código. Internamente usa PyTorch, que es la biblioteca de cálculo.

La instalación toma alrededor de un minuto y produce bastante texto de diagnóstico que pueden
ignorar.

In [ ]:
!pip install -Uqq fastai

In [ ]:
from fastai.vision.all import *   # DataBlock, Learner, Resize, aug_transforms, ...
from torch import nn              # las piezas con las que se construyen redes neuronales

import urllib.request, zipfile, random
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

print("fastai", __import__('fastai').__version__)

## 1. Los datos

En la clase las imágenes se descargaban. Aquí son **nuestras fotos**, así que no hay descarga:
se leen desde una carpeta (por ejemplo, en Google Drive).

La respuesta correcta de cada imagen **no está escrita en ningún archivo**: está en el nombre de la
carpeta que la contiene. A cada categoría la llamamos **clase** y a la clase correcta de una
imagen, su **etiqueta**.

### Opcional: conectar Google Drive (Colab)
Si las fotos están en Drive, ejecutar la celda siguiente y aceptar el permiso. Si las fotos se
subieron directamente a Colab (panel de archivos), se puede omitir.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("No estamos en Colab: se omite el montaje de Drive.")

### Opcional: reducir las fotos
Una foto de celular pesa ~3 MB y la red solo usa 128 píxeles de lado, así que conviene reducirlas
antes (el anexo del enunciado). Esta celda copia cada foto de `ORIGINALES` a `REDUCIDAS`
conservando la subcarpeta, a un máximo de 256 × 256. **Comprobar después una original y su
reducida lado a lado: ¿se ven equivalentes?**

In [ ]:
# ⚙ PARÁMETROS (solo si se van a reducir las fotos; si ya vienen reducidas, omitir esta celda)
from PIL import Image

ORIGINALES = Path("/content/drive/MyDrive/mis_fotos")   # carpeta con las fotos originales
REDUCIDAS  = Path("fotos")                              # carpeta de salida

for foto in ORIGINALES.rglob("*"):
    if foto.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    destino = REDUCIDAS / foto.relative_to(ORIGINALES).parent          # conserva entrenamiento|prueba / objeto
    destino.mkdir(parents=True, exist_ok=True)
    img = Image.open(foto).convert('RGB')
    img.thumbnail((256, 256))
    img.save(destino / (foto.stem + ".jpg"), quality=85)

print("Listo. Usar CARPETA_ENTRENAMIENTO = Path('fotos/entrenamiento')")

### Dónde están las carpetas

In [ ]:
# ⚙ PARÁMETROS: carpetas con las fotos (una subcarpeta por objeto)
CARPETA_ENTRENAMIENTO = Path("fotos/entrenamiento")
CARPETA_PRUEBA        = Path("fotos/prueba")

# get_image_files recorre la carpeta y sus subcarpetas y devuelve la lista de rutas de imagen.
imagenes = get_image_files(CARPETA_ENTRENAMIENTO)

print("Total de imagenes de entrenamiento:", len(imagenes))
print()
print("Ruta de una imagen cualquiera:")
print("  ", imagenes[0])

# parent_label recibe una ruta y devuelve el nombre de su carpeta contenedora: la etiqueta.
print("Su etiqueta:", parent_label(imagenes[0]))

Antes de entrenar cualquier cosa hay que verificar que las clases estén **balanceadas**, es decir
que haya una cantidad comparable de ejemplos de cada una.

La razón es concreta. Si el 95 % de las imágenes fueran de piedra, existiría una estrategia
trivial que acierta el 95 % de las veces: responder siempre "piedra". El modelo podría converger
a esa estrategia y su tasa de acierto se vería excelente sin que haya aprendido absolutamente
nada. Con clases balanceadas esa salida no existe.

In [ ]:
conteo = Counter(parent_label(f) for f in imagenes)

clases  = sorted(conteo)
valores = [conteo[c] for c in clases]

plt.figure(figsize=(7, 4))
plt.bar(clases, valores, color='steelblue', width=0.5)
plt.ylabel('número de imágenes')
plt.title('Distribución de las clases')
plt.ylim(0, max(valores)*1.15)
for c, v in zip(clases, valores):
    plt.text(c, v, str(v), ha='center', va='bottom')
plt.show()

In [ ]:
# Regla general: nunca se entrena un modelo sin haber mirado los datos primero.
ejemplo = imagenes[0]
img = PILImage.create(ejemplo)

print("Etiqueta:", parent_label(ejemplo))
print("Resolución:", img.size)

img.show()

### Qué recibe realmente el modelo

Donde ustedes ven una mano, el modelo recibe una tabla de números.

Una imagen está formada por **píxeles**, y cada píxel es un número que indica intensidad: 0 es
negro, 255 es blanco, y los valores intermedios son grises. Como la imagen es a color, cada píxel
necesita tres de esos números (cuánto rojo, cuánto verde, cuánto azul), así que una imagen de
300 × 300 son 270.000 números.

Eso es todo lo que entra al modelo. No hay información adicional, ni contexto, ni sentido común.
La celda siguiente muestra un recorte de 6 × 6 píxeles del centro de la imagen tal como el modelo
lo recibe.

In [ ]:
valores_pixel = image2tensor(img)   # convierte la imagen en un arreglo numerico

print("Dimensiones del arreglo:", tuple(valores_pixel.shape), " (canales, alto, ancho)")
print("Valor mínimo:", valores_pixel.min().item(), "  máximo:", valores_pixel.max().item())
print()
alto, ancho = valores_pixel.shape[1], valores_pixel.shape[2]
cy, cx = alto // 2, ancho // 2          # el centro, sea cual sea el tamaño de la foto
print("Recorte de 6x6 pixeles del centro, canal rojo:")
print(valores_pixel[0, cy-3:cy+3, cx-3:cx+3])

# Los tres canales de color, por separado
fig, axs = plt.subplots(1, 3, figsize=(11, 4))
for ax, k, nombre in zip(axs, range(3), ['rojo', 'verde', 'azul']):
    ax.imshow(valores_pixel[k], cmap='gray')
    ax.set_title(nombre); ax.axis('off')
plt.show()

## 2. Separar los datos: entrenamiento y validación

Aquí hay una decisión metodológica que determina si los resultados significan algo.

Si midiéramos el desempeño del modelo con las mismas imágenes que usó para aprender, estaríamos
midiendo **memorización**, no capacidad de reconocer. Un modelo con suficientes parámetros puede
almacenar los ejemplos que vio y responder perfecto sobre ellos sin funcionar con nada nuevo.

Por eso se parte el conjunto en dos:

- **Conjunto de entrenamiento** (80 %, unas 2.016 imágenes): con estas el modelo ajusta sus
  parámetros.
- **Conjunto de validación** (20 %, unas 504): el modelo **nunca las usa para aprender**. Solo
  sirven para medir.

Cuando el desempeño sobre entrenamiento sigue mejorando pero sobre validación se estanca o
empeora, el modelo está memorizando en lugar de generalizar. Eso se llama **sobreajuste**, y es
el problema más común en la práctica.

> **Con nuestros datos son pocas imágenes.** 180 fotos → unas **144 de entrenamiento y 36 de
> validación**. La validación es ruidosa: cada imagen mal clasificada mueve el accuracy unos
> 2,8 puntos. Por eso las fotos de **prueba** (aparte) importan tanto.
>
> **Y también hay pocos ajustes.** En la clase había 2.016 imágenes de entrenamiento: con lotes
> de 32 eran ~63 ajustes por época. Con ~144 imágenes y lotes de 32 son **solo 4 por época (48 en
> 12 épocas)**, y eso no alcanzó en un ensayo con datos de sustitución: el modelo terminó
> respondiendo siempre la misma clase (33 %). Por eso aquí el lote es **`BS = 16`** y las épocas
> son **30**: más ajustes en total. (Los valores de la clase eran 32 y 12.)

In [ ]:
# ⚙ PARÁMETROS
USAR_AUMENTO = True   # experimento «sin aumento de datos»: True = con aumento (base), False = sin aumento
BS = 16               # tamaño del lote: cuántas imágenes se procesan antes de cada ajuste (ver la nota de arriba)

datos = DataBlock(
    # Tipo de entrada y tipo de salida: una imagen entra, una clase sale.
    blocks=(ImageBlock, CategoryBlock),

    # Cómo obtener la lista de elementos.
    get_items=get_image_files,

    # Cómo obtener la etiqueta de cada elemento.
    get_y=parent_label,

    # Partición entrenamiento / validación. seed fija el sorteo: con el mismo valor,
    # la partición es siempre la misma y los resultados son reproducibles.
    splitter=RandomSplitter(valid_pct=0.2, seed=42),

    # Se aplica a cada imagen por separado. Todas deben quedar del mismo tamaño para
    # poder agruparlas en un mismo arreglo.
    item_tfms=Resize(160) if USAR_AUMENTO else Resize(128),

    # Se aplica al grupo completo, en la GPU: aquí vive el aumento de datos.
    # Sin aumento (experimento) las imágenes solo se dejan a 128 px, igual que en el caso base.
    batch_tfms=aug_transforms(size=128) if USAR_AUMENTO else [],
)

# bs (batch size) es cuántas imágenes se procesan juntas; el modelo se ajusta una vez por lote.
dls = datos.dataloaders(CARPETA_ENTRENAMIENTO, bs=BS)

print("Clases:", list(dls.vocab))
print("Entrenamiento:", len(dls.train_ds), " Validación:", len(dls.valid_ds))

In [ ]:
# Inspeccionar un grupo de imagenes antes de entrenar verifica dos cosas simultaneamente:
#   1. que las etiquetas correspondan a lo que se ve (el pipeline de datos es correcto)
#   2. que las transformaciones no esten destruyendo la informacion
dls.show_batch(max_n=8, nrows=2)

## 3. La red neuronal convolucional

Ahora el modelo. Vamos a construir una **red neuronal convolucional**, abreviada CNN, que es la
arquitectura estándar para imágenes.

### La operación básica: el filtro

Un **filtro** es un detector de patrones pequeño (3 × 3 píxeles) que se desliza sobre toda la
imagen comparándose con cada posición. Donde encuentra el patrón que busca, produce un valor
alto; donde no, un valor bajo. El resultado es un **mapa de activación**: una imagen nueva que
indica en qué lugares apareció ese patrón.

Un mismo filtro recorre toda la imagen, y esa es precisamente la ventaja de la convolución: un
detector de bordes sirve igual en la esquina superior izquierda que en el centro, así que no hace
falta aprender uno distinto para cada posición.

### Apilar capas

Una **capa convolucional** aplica varios filtros a la vez. Nuestra red tiene cuatro capas en
secuencia, y lo relevante es que **cada capa no mira la imagen: mira la salida de la capa
anterior**.

| capa | qué recibe | qué detecta |
|---|---|---|
| 1 | la imagen | bordes, cambios de color, rayas |
| 2 | los bordes de la capa 1 | esquinas, curvas, cruces |
| 3 | las esquinas de la capa 2 | fragmentos reconocibles: la punta de un dedo, un contorno cerrado |
| 4 | los fragmentos de la capa 3 | configuraciones completas: dos dedos extendidos y tres flexionados |

Esa composición es la razón de la profundidad. Con una sola capa el modelo solo puede responder a
patrones de 3 × 3 píxeles, y ninguna mano cabe en 3 × 3. Cada capa adicional amplía la región de
la imagen que un detector alcanza a tener en cuenta.

### Lo que no hacemos

**En ningún momento especificamos qué debe buscar cada filtro.** Los filtros arrancan con valores
aleatorios y, por lo tanto, detectando nada en particular. El entrenamiento consiste en ajustarlos
progresivamente: cada vez que el modelo se equivoca, sus parámetros se modifican un poco en la
dirección que habría reducido el error. Repetido miles de veces, el proceso converge a filtros que
sí detectan estructuras útiles.

Nadie codificó el concepto de borde ni el de dedo. Emergen del ajuste.

### Las cuatro piezas de cada capa

Cada capa de nuestra red es la misma secuencia de cuatro operaciones:

| pieza | función | por qué está |
|---|---|---|
| `Conv2d` | aplica los filtros | es la detección de patrones propiamente dicha |
| `BatchNorm2d` | reescala los resultados a un rango comparable | sin esto el entrenamiento es inestable y hay que afinar mucho más la tasa de aprendizaje |
| `ReLU` | pone en cero los valores negativos | es la operación no lineal; sin ella, apilar capas no aporta nada, porque la composición de operaciones lineales es otra operación lineal |
| `MaxPool2d` | reduce la resolución a la mitad, quedándose con el valor más alto de cada región | amplía lo que la siguiente capa alcanza a ver y reduce el costo de cómputo |

Al avanzar por la red, la resolución baja (128 → 64 → 32 → 16 → 8) mientras el número de filtros
sube (3 → 16 → 32 → 64 → 128). Se pierde precisión sobre *dónde* está cada cosa y se gana
información sobre *qué* es.

In [ ]:
def capa(entran, salen):
    # Una capa convolucional completa. "salen" es cuantos filtros tiene.
    return nn.Sequential(
        nn.Conv2d(entran, salen, kernel_size=3, padding=1),  # los filtros de 3x3
        nn.BatchNorm2d(salen),                               # reescala los resultados
        nn.ReLU(),                                           # pone en cero los negativos
        nn.MaxPool2d(2),                                     # reduce la resolucion a la mitad
    )


def crear_red(n_clases=3):   # 3 objetos = 3 clases
    return nn.Sequential(
        capa(3,   16),     # 128 -> 64 px,  16 filtros
        capa(16,  32),     #  64 -> 32 px,  32 filtros
        capa(32,  64),     #  32 -> 16 px,  64 filtros
        capa(64, 128),     #  16 ->  8 px, 128 filtros

        nn.AdaptiveAvgPool2d(1),   # resume cada mapa de activacion en un solo numero
        nn.Flatten(),              # deja un vector de 128 numeros
        nn.Linear(128, n_clases),  # la decision final: una puntuacion por clase
    )


red = crear_red()

print("Parámetros ajustables:", sum(p.numel() for p in red.parameters()))
print()

# Verificacion de dimensiones: pasamos un grupo de imagenes y vemos como cambia en cada paso.
x, _ = dls.one_batch()
z = x[:4].cpu()
print("entrada           ", tuple(z.shape))
for i, parte in enumerate(red):
    z = parte(z)
    print("después del paso", i, tuple(z.shape))

## 4. Entrenamiento

Entrenar consiste en recorrer el conjunto de entrenamiento muchas veces. Cada recorrido completo
se llama una **época**. En cada época el modelo procesa las imágenes en grupos, compara sus
respuestas con las etiquetas correctas y ajusta sus parámetros.

En la clase eran 12 épocas; con nuestro conjunto más pequeño usamos **30** (variable `EPOCAS`). Durante el proceso aparece una tabla; estas son las columnas que
importan:

| columna | qué es | qué esperamos |
|---|---|---|
| `train_loss` | magnitud del error sobre el conjunto de entrenamiento | que baje |
| `valid_loss` | magnitud del error sobre validación | que baje también |
| `accuracy` | proporción de acierto sobre validación | que suba |

El punto de referencia: con tres clases, **responder al azar acierta una de cada tres veces**. Un
33 % no es un resultado, es el punto de partida.

Y la señal de alarma: si `train_loss` sigue bajando mientras `valid_loss` empieza a subir, el
modelo está sobreajustando (memorizando el conjunto de entrenamiento) y entrenar más lo empeora.

Esta celda toma alrededor de 3 minutos.

In [ ]:
# ⚙ PARÁMETROS de los experimentos: cada experimento cambia UNO de estos valores (o una pieza
# de la red / del DataBlock) y se vuelve a ejecutar desde aquí.
EPOCAS = 30      # cuántas veces se recorre el conjunto de entrenamiento (en la clase eran 12)
TASA   = 3e-3    # tasa de aprendizaje: cuánto se mueven los parámetros en cada ajuste

# Learner reune los datos, el modelo y la funcion que mide el error.
#   CrossEntropyLossFlat es la medida de error estandar en clasificacion: penaliza segun
#   la probabilidad que el modelo le asigno a la clase correcta.
learn = Learner(dls, crear_red(len(dls.vocab)), loss_func=CrossEntropyLossFlat(), metrics=accuracy)

# fit_one_cycle hace EPOCAS epocas. El segundo argumento es la tasa de aprendizaje: cuanto se
# mueven los parametros en cada ajuste. Demasiado alta y el entrenamiento se desestabiliza;
# demasiado baja y no avanza.
learn.fit_one_cycle(EPOCAS, TASA)

In [ ]:
registro = learn.recorder.values      # [[train_loss, valid_loss, accuracy], ...]

train = [v[0] for v in registro]
valid = [v[1] for v in registro]
acc   = [v[2] for v in registro]
ep    = range(1, len(train)+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(ep, train, '-o', label='entrenamiento', color='steelblue')
ax1.plot(ep, valid, '-s', label='validación', color='crimson')
ax1.set_xlabel('época'); ax1.set_ylabel('error'); ax1.set_yscale('log')
ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(ep, acc, '-o', color='darkmagenta')
ax2.axhline(1/3, color='gray', linestyle='--')
ax2.text(1, 1/3 + 0.02, 'nivel del azar', color='gray', fontsize=10)
ax2.set_xlabel('época'); ax2.set_ylabel('accuracy'); ax2.set_ylim(0, 1.05)
ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()

### Los filtros que resultaron del entrenamiento

Vale la pena mirar qué terminaron detectando los 16 filtros de la primera capa, teniendo en cuenta
que empezaron con valores aleatorios y que nadie especificó qué debían buscar.

Cada cuadro de abajo es un filtro de 3 × 3 dibujado como imagen. No van a ver dedos ni manos: van
a ver gradientes, bordes y manchas de color. 

In [ ]:
filtros = learn.model[0][0].weight.detach().cpu()   # dimensiones: (16, 3, 3, 3)

print("Dimensiones:", tuple(filtros.shape), " (filtros, canales, alto, ancho)")

fig, axs = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, k in zip(axs.flat, range(filtros.shape[0])):
    f = filtros[k]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)   # reescala a [0,1] para poder dibujarlo
    ax.imshow(f.permute(1, 2, 0)); ax.axis('off')
fig.suptitle('Los 16 filtros de la primera capa')
plt.show()

## 5. Evaluación

Ya tenemos una cifra de acierto. Pero una cifra global no dice **cómo** se distribuyen los
errores, y esa información es la que permite mejorar el modelo.

In [ ]:
# Predicciones individuales. Cada imagen se rotula "etiqueta real / prediccion";
# cuando no coinciden, fastai lo marca en rojo.
learn.show_results(max_n=6, nrows=2)

### Matriz de confusión

La matriz de confusión se lee así: cada **fila** es la clase real de las imágenes, cada
**columna** es la clase que el modelo predijo. Las casillas de la diagonal son aciertos; todo lo
que esté fuera de la diagonal es un error, y su posición indica exactamente qué clase se confundió
con cuál.

Es la herramienta de diagnóstico más informativa en clasificación, porque dos modelos con la misma
tasa de acierto pueden tener errores completamente distintos: repartidos entre todas las clases, o
concentrados en un único par. Son dos situaciones que se corrigen de forma diferente.

In [ ]:
analisis = ClassificationInterpretation.from_learner(learn)

analisis.plot_confusion_matrix(figsize=(5.5, 5.5), cmap='Blues')

In [ ]:
# Los errores en los que el modelo estaba mas seguro de si mismo. Son los casos que mas
# conviene revisar: con frecuencia revelan etiquetas mal asignadas en los datos, no fallos
# del modelo.
# Cada imagen se rotula: prediccion / etiqueta real / error / probabilidad de la clase correcta
analisis.plot_top_losses(4, nrows=2, figsize=(9, 6))

### El conjunto de prueba

Un detalle metodológico que se pasa por alto con frecuencia.

Medimos sobre el conjunto de validación, pero ese conjunto lo usamos para decidir cuántas épocas
entrenar, qué tasa de aprendizaje usar y cuántas capas poner. Cada una de esas decisiones se tomó
mirando el resultado en validación, así que el conjunto quedó indirectamente involucrado en el
ajuste y su medida ya está optimista.

Para eso existe un tercer conjunto, el **conjunto de prueba**, que no se toca en ningún momento
del proceso. Este dataset trae uno aparte, con 372 imágenes. Esa es la única estimación limpia.

In [ ]:
imagenes_prueba = get_image_files(CARPETA_PRUEBA)

# test_dl construye un cargador con imagenes nuevas aplicando el mismo preprocesamiento;
# with_labels=True extrae tambien la etiqueta real para poder comparar.
prueba = learn.dls.test_dl(imagenes_prueba, with_labels=True)

error, acierto = learn.validate(dl=prueba)

print("Imágenes de prueba:", len(imagenes_prueba))
print("Accuracy sobre datos nunca vistos: {:.2%}".format(acierto))

### Matriz de confusión sobre el conjunto de prueba
La matriz anterior era sobre validación. Esta es sobre las **fotos de prueba** (las que se tomaron
en otras condiciones): es la que responde *¿qué par de objetos confunde el modelo y por qué?*

In [ ]:
analisis_prueba = ClassificationInterpretation.from_learner(learn, dl=prueba)
analisis_prueba.plot_confusion_matrix(figsize=(5.5, 5.5), cmap='Blues')

# Los pares que mas se confunden, de mayor a menor: (real, predicho, cuantas veces)
print("Pares más confundidos:", analisis_prueba.most_confused())

## 6. Prueba en vivo (para el video)

El enunciado pide pasar **5 imágenes del conjunto de prueba por el modelo mientras se graba, sin
cortes**, y comentar qué responde en cada una. **Si el modelo falla, se muestra y se explica:** no
se repite hasta que salga bien.

La celda muestra cada imagen, la clase que predice el modelo, la clase real y qué tan seguro
estaba (la probabilidad que asignó a su respuesta).

In [ ]:
# ⚙ PARÁMETRO: para elegir las imagenes a mano, reemplazar por una lista de rutas, p. ej.
#   elegidas = [CARPETA_PRUEBA/"objeto1"/"foto01.jpg", ...]
elegidas = random.sample(list(imagenes_prueba), 5)

for ruta in elegidas:
    # learn.predict devuelve (clase predicha, indice de la clase, probabilidad de cada clase)
    prediccion, indice, probabilidades = learn.predict(ruta)
    real = parent_label(ruta)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(PILImage.create(ruta).to_thumb(224))        # la imagen se dibuja en el eje ax
    ax.set_title("real: {}  →  predice: {}  ({:.0%} seguro)  {}".format(
        real, prediccion, probabilidades[indice], "✔" if prediccion == real else "✘ ERROR"), fontsize=10)
    ax.axis('off')
    plt.show()